# Supervised Fine-Tuning (SFT) with Serverless Customization on SageMaker AI

## Lab 4 – Deploy the Fine-Tuned Model to a SageMaker Endpoint

> ⚠️ **Compatibility Notice:** This workshop has been tested with the **SageMaker Distribution Image 2.5.0** and **SageMaker Python SDK v2.x**. If you encounter issues, ensure your kernel is set to **Python 3 (ipykernel)**.

> **Prerequisite:** Complete **Lab 2** (`2-fine-tune-llm.ipynb`) before running this notebook. The fine-tuned model must be registered in the SageMaker Model Registry with a merged model artifact available in S3.

This is the final lab in the series:

| Lab | Notebook | What you'll do |
|-----|----------|----------------|
| **Lab 1** | `1-prepare-data.ipynb` | Prepare and register the dataset |
| **Lab 2** | `2-fine-tune-llm.ipynb` | Submit serverless fine-tuning and register the result |
| **Lab 3** | `3-evaluation.ipynb` | Evaluate the fine-tuned model with LLM-as-Judge |
| **Lab 4** | `4-deployment.ipynb` ← *you are here* | Deploy the merged model to a real-time SageMaker endpoint |

### What you'll do in this lab

1. **Retrieve the merged fine-tuned model** artifact from the Model Registry
2. **Configure a SageMaker real-time endpoint** on an `ml.g5.12xlarge` GPU instance
3. **Deploy via vLLM** with a custom `nano_v3` reasoning parser that handles Nemotron's `<think>` format
4. **Create an Inference Component** for efficient, multi-model hosting
5. **Test the endpoint** with a streaming inference request
6. **Clean up** all AWS resources to avoid ongoing costs

### Architecture overview

The model was fine-tuned on the [Multilingual-Thinking](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking) dataset. At inference time, it reasons inside `<think>…</think>` tags in a target non-English language (set via the system prompt) and answers in English.

```
Fine-tuned model weights (S3)
          ↓
    SageMaker Model
          ↓
 Inference Component  →  SageMaker Endpoint (ml.g5.12xlarge, 4× A10G GPUs)
                                   ↓
                     vLLM + nano_v3 reasoning parser
                                   ↓
                      OpenAI-compatible streaming API
                         ├── reasoning_content  (the <think> block)
                         └── content            (the English final answer)
```

***

In [ ]:
%load_ext autoreload
%autoreload 2

### Prerequisites

Before running this notebook, confirm that:
- **Lab 2 is complete** — a fine-tuned model package is registered in the SageMaker Model Registry with a merged model artifact in S3
- Your IAM execution role has `AmazonSageMakerFullAccess` and S3 read/write permissions
- The `ml.g5.12xlarge` instance type is available in your AWS region (4× NVIDIA A10G GPUs required for tensor-parallel inference of the 30B model)

### Step 1 – Set up the SageMaker session

#### Setup and dependencies

We re-establish the SageMaker session and pre-compute all resource names (model, endpoint config, endpoint, inference component). SageMaker enforces a 63-character limit on resource names, so long model IDs are truncated with a short hash to avoid collisions while remaining deterministic.

In [ ]:
import boto3
import os
from rich.pretty import pprint
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

### Step 2 – Retrieve the fine-tuned model

After LoRA fine-tuning, SageMaker automatically **merges** the adapter weights back into the base model and saves the result as a single set of model weights in S3 (under the `checkpoints/hf_merged/` path within the model package). 

This merged model is what we deploy — it runs as a standard dense model without any adapter overhead, giving you full inference performance. We retrieve the S3 URI of these merged weights from the Model Registry.

In [ ]:
import hashlib

from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID

MAX_NAME_LENGTH = 63  # SageMaker resource name limit


def build_resource_name(base, suffix, max_length=MAX_NAME_LENGTH):
    candidate = f"{base}{suffix}"
    if len(candidate) <= max_length:
        return candidate
    digest = hashlib.sha1(base.encode()).hexdigest()[:6]
    # reserve room for the suffix, a hyphen separator, and the 6-char hash
    keep = max_length - len(suffix) - len(digest) - 1
    truncated = base[:keep].rstrip("-")
    return f"{truncated}-{digest}{suffix}"


model_name = build_resource_name(base_model_id, "-sft")
endpoint_config_name = build_resource_name(base_model_id, "-sft-config")
endpoint_name = build_resource_name(base_model_id, "-sft-endpoint")
ic_name = build_resource_name(base_model_id, "-sft-ic")

In [ ]:
import hashlib

MAX_MPG_NAME_LENGTH = 63
suffix = "-sft-mpg"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    # reserve room for the suffix, a hyphen separator, and the 6-char hash
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

In [ ]:
from sagemaker.core import s3
from sagemaker.core.resources import ModelPackage, ModelPackageGroup

response = sm_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1,
)

if len(response["ModelPackageSummaryList"]) > 0:
    fine_tuned_model_package_arn = response["ModelPackageSummaryList"][0][
        "ModelPackageArn"
    ]
    fine_tuned_model_package_group_arn = ModelPackageGroup.get(
        model_package_group_name
    ).model_package_group_arn

    model_package = ModelPackage.get(fine_tuned_model_package_arn)

    # get the merged model artifact and deploy it
    merged_model_s3_uri = (
        s3.s3_path_join(
            model_package.inference_specification.containers[
                0
            ].model_data_source.s3_data_source.s3_uri,
            "checkpoints",
            "hf_merged",
        )
        + "/"
    )
else:
    fine_tuned_model_package_arn = None
    fine_tuned_model_package_group_arn = None
    merged_model_s3_uri = None

print(f"Model S3 path: {merged_model_s3_uri}")

***

### Step 3 – Create an Endpoint Configuration

An **EndpointConfig** defines the infrastructure blueprint for a SageMaker endpoint — which instance type to use, how many copies to run, and how to route traffic. The endpoint itself (created in the next step) is a persistent HTTPS endpoint that stays running and ready to serve requests.

Key settings for this lab:

| Setting | Value | Why |
|---------|-------|-----|
| `instance_type` | `ml.g5.12xlarge` | 4× NVIDIA A10G GPUs — needed to fit the 30B model with tensor parallelism |
| `initial_instance_count` | `1` | Single instance for the workshop; scale up for production traffic |
| `inference_ami_version` | `al2-ami-sagemaker-inference-gpu-3-1` | Latest GPU-optimized AMI with current CUDA drivers |
| `routing_strategy` | `LEAST_OUTSTANDING_REQUESTS` | Routes each new request to the instance with the fewest in-flight requests, reducing latency under load |

Define inference configuration

In [ ]:
instance_count = 1
instance_type = "ml.g5.12xlarge"
number_of_gpu = 4  # ml.g5.12xlarge has 4x A10G GPUs -> vLLM tensor parallel size
health_check_timeout = 700

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig
from sagemaker.core.shapes import ProductionVariant

print(f"Creating EndpointConfig: {endpoint_config_name}")
endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    execution_role_arn=role,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            instance_type=instance_type,
            initial_instance_count=instance_count,
            model_data_download_timeout_in_seconds=health_check_timeout,
            inference_ami_version="al2-ami-sagemaker-inference-gpu-3-1",
            routing_config={"routing_strategy": "LEAST_OUTSTANDING_REQUESTS"},
        )
    ],
)

### Step 4 – Create the SageMaker Endpoint

A **SageMaker Endpoint** is a fully managed, always-on HTTPS API that hosts your model and serves real-time inference requests. Once in service, it remains running (and billable) until you delete it.

> ⏳ Endpoint creation typically takes **5–10 minutes**. The cell below blocks until the endpoint reaches `InService` status. You can also monitor progress in the AWS console under **SageMaker AI → Inference → Endpoints**.

In [ ]:
print(f"Creating Endpoint: {endpoint_name}")
endpoint = Endpoint.create(
    endpoint_name=endpoint_name, endpoint_config_name=endpoint_config_name
)
endpoint.wait_for_status("InService")
print(f"Endpoint {endpoint_name} is InService")

### Step 5 – Create the Model and configure vLLM

We deploy the fine-tuned model using the **vLLM SageMaker Deep Learning Container (DLC)** — a high-performance inference engine optimized for transformer-based LLMs. vLLM provides:

- **Continuous batching** — processes new requests as soon as compute frees up, maximizing GPU utilization
- **PagedAttention** — efficient KV-cache memory management that reduces memory fragmentation and supports longer context windows
- **OpenAI-compatible API** — your existing OpenAI SDK integrations work without modification
- **Tensor parallelism** — automatically shards the model across all 4 GPUs on the instance

#### Custom `nano_v3` reasoning parser

The fine-tuned Nemotron model uses `<think>…</think>` tags for its chain-of-thought. We upload a custom vLLM **reasoning parser plugin** (`nano_v3_reasoning_parser.py`) alongside the model weights so vLLM knows how to extract and route this content.

The parser:
- When `enable_thinking=True` (default): extracts `<think>…</think>` content into `reasoning_content` in the API response, separate from the final `content`
- When `enable_thinking=False`: returns the model's output directly as `content`, without explicit reasoning tags

This gives API callers full control — they can display the thinking process, hide it, or use it for downstream reasoning pipelines.

> 📖 Learn more: [vLLM documentation](https://docs.vllm.ai/en/latest/) · [SageMaker Deep Learning Containers](https://docs.aws.amazon.com/sagemaker/latest/dg/pre-built-containers-frameworks-deep-learning.html)

Get the image URI

In [ ]:
region = sess.boto_region_name

CONTAINER_VERSION = "vllm:0.22.0-gpu-py312-cu130-ubuntu22.04-sagemaker"

inference_image = (
    f"763104351884.dkr.ecr.{region}.amazonaws.com/{CONTAINER_VERSION}"
)

#### Custom `nano_v3` reasoning parser plugin

The `nano_v3` parser extends vLLM's built-in `DeepSeekR1ReasoningParser` to handle Nemotron's specific behavior. It is registered with vLLM via the `@ReasoningParserManager.register_module("nano_v3")` decorator and referenced in `vllm_config.yaml`.

Both the parser file and the vLLM config YAML are uploaded directly into the model's S3 prefix so vLLM loads them automatically when the container starts.

In [ ]:
reasoning_parser_file = "nano_v3_reasoning_parser.py"

nano_v3_reasoning_parser = '''from vllm.reasoning.abs_reasoning_parsers import ReasoningParserManager
from vllm.reasoning.deepseek_r1_reasoning_parser import DeepSeekR1ReasoningParser


@ReasoningParserManager.register_module("nano_v3")
class NanoV3ReasoningParser(DeepSeekR1ReasoningParser):
    def extract_reasoning(self, model_output, request):
        reasoning_content, final_content = super().extract_reasoning(
            model_output, request
        )
        if (
            hasattr(request, "chat_template_kwargs")
            and request.chat_template_kwargs
            and request.chat_template_kwargs.get("enable_thinking") is False
            and final_content is None
        ):
            reasoning_content, final_content = final_content, reasoning_content

        return reasoning_content, final_content
'''

with open(reasoning_parser_file, "w") as f:
    f.write(nano_v3_reasoning_parser)

#### vLLM configuration file

The `vllm_config.yaml` file configures the vLLM server. Key settings:

| Setting | Value | What it does |
|---------|-------|-------------|
| `enable_auto_tool_choice` | `true` | Enables OpenAI-compatible function/tool calling |
| `tool_call_parser` | `qwen3_coder` | Parser for structured tool call outputs |
| `reasoning_parser` | `nano_v3` | Activates our custom reasoning parser |
| `reasoning_parser_plugin` | `/opt/ml/model/nano_v3_reasoning_parser.py` | Path to the parser plugin inside the container |

In [ ]:
vllm_config_file = "vllm_config.yaml"

vllm_config = """enable_auto_tool_choice: true
tool_call_parser: qwen3_coder
reasoning_parser: nano_v3
reasoning_parser_plugin: /opt/ml/model/nano_v3_reasoning_parser.py
"""

with open(vllm_config_file, "w") as f:
    f.write(vllm_config)

Upload files in the S3 model path

In [ ]:
import os
from urllib.parse import urlparse

parsed = urlparse(merged_model_s3_uri)
model_bucket = parsed.netloc
model_prefix = parsed.path.lstrip("/").rstrip("/")

s3_client.upload_file(
    reasoning_parser_file, model_bucket, f"{model_prefix}/{reasoning_parser_file}"
)
s3_client.upload_file(
    vllm_config_file, model_bucket, f"{model_prefix}/{vllm_config_file}"
)

os.remove(reasoning_parser_file)
os.remove(vllm_config_file)

print("Uploaded vLLM config + reasoning parser to:")
print(f"s3://{model_bucket}/{model_prefix}/{reasoning_parser_file}")
print(f"s3://{model_bucket}/{model_prefix}/{vllm_config_file}")

In [ ]:
import json

env = {
    "SM_VLLM_MODEL": "/opt/ml/model",  # path where SageMaker mounts the model
    "SM_VLLM_CONFIG": "/opt/ml/model/vllm_config.yaml",
    "SM_VLLM_DTYPE": "bfloat16",
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.9",
    "SM_VLLM_MAX_MODEL_LEN": json.dumps(1024 * 32),
    "SM_VLLM_MAX_NUM_SEQS": "16",
    "SM_VLLM_ENABLE_CHUNKED_PREFILL": "true",
    "SM_VLLM_KV_CACHE_DTYPE": "auto",
    "SM_VLLM_TENSOR_PARALLEL_SIZE": str(number_of_gpu),
}

In [ ]:
from sagemaker.core.resources import Model
from sagemaker.core.shapes import (
    ContainerDefinition,
    ModelDataSource,
    S3ModelDataSource,
)

fine_tuned_model = Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        image=inference_image,
        model_data_source=ModelDataSource(
            s3_data_source=S3ModelDataSource(
                s3_uri=merged_model_s3_uri,
                s3_data_type="S3Prefix",
                compression_type="None",
            )
        ),
        environment=env,
    ),
    execution_role_arn=role,
)

pprint(fine_tuned_model)

### Step 6 – Create an Inference Component

An **Inference Component** is a SageMaker construct that decouples the *model* from the *endpoint infrastructure*. Rather than dedicating one model per endpoint, you can host multiple models on the same endpoint — each as its own Inference Component — with independent scaling controls.

Benefits for this lab and beyond:
- **Resource efficiency** — share one `ml.g5.12xlarge` instance across multiple models
- **Independent copy scaling** — increase `copy_count` to run multiple replicas of the same model for higher throughput
- **Clean rollout** — swap model versions by updating the Inference Component without recreating the endpoint

> ⏳ The Inference Component takes **5–10 minutes** to reach `InService`. The vLLM server downloads model weights from S3 and loads them across all GPUs during this time.

> 📖 Learn more: [Amazon SageMaker Inference Components](https://docs.aws.amazon.com/sagemaker/latest/dg/inference-component.html)

In [ ]:
from sagemaker.core.resources import InferenceComponent
from sagemaker.core.shapes import (
    InferenceComponentSpecification,
    InferenceComponentComputeResourceRequirements,
    InferenceComponentRuntimeConfig,
)

# Step 3: Create InferenceComponent
inference_component = InferenceComponent.create(
    inference_component_name=ic_name,
    endpoint_name=endpoint_name,
    variant_name="AllTraffic",
    specification=InferenceComponentSpecification(
        model_name=model_name,
        compute_resource_requirements=InferenceComponentComputeResourceRequirements(
            min_memory_required_in_mb=10240,
            number_of_accelerator_devices_required=number_of_gpu,
        ),
    ),
    runtime_config=InferenceComponentRuntimeConfig(copy_count=1),
    region=region,
)

print(f"InferenceComponent created: {inference_component.inference_component_name}")
print(f"Endpoint ARN: {endpoint.endpoint_arn}")
inference_component.wait_for_status("InService")
print(f"Endpoint {ic_name} is InService")

***

### Step 7 – Test the endpoint

We send a **streaming** inference request to the endpoint. The request:
1. Sets a system prompt instructing the model to reason in **Italian**
2. Asks a factual question in English
3. Streams the response token-by-token, with the `nano_v3` reasoning parser separating the `<think>` block (`reasoning_content`) from the final English answer (`content`)

**Why streaming?** For long reasoning responses, streaming lets you begin displaying output to the user before generation is complete, dramatically reducing perceived latency. The `LineIterator` helper class below handles the server-sent events (SSE) format used by the vLLM streaming API.

In [ ]:
import io
import json
import boto3

In [ ]:
sagemaker_client = boto3.client(service_name="sagemaker-runtime")

### Iterator class for streaming inference

Utility class to parse streaming responses

In [ ]:
class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()

            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]

            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise

            if "PayloadPart" not in chunk:
                continue

            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

Utility function to parse model answer

In [ ]:
def parse_streaming_response(line_str):
    """Parse a streaming response line and return (reasoning, content, tool_calls)."""
    if not line_str.strip() or line_str.strip() == "data: [DONE]":
        return None, None, None

    if line_str.startswith("data: "):
        line_str = line_str[6:]

    try:
        data = json.loads(line_str)
        if "choices" in data:
            for choice in data["choices"]:
                delta = choice.get("delta", {})
                reasoning = delta.get("reasoning") or delta.get("reasoning_content")
                content = delta.get("content")
                tool_calls = delta.get("tool_calls")
                if reasoning or content or tool_calls:
                    return reasoning, content, tool_calls
    except json.JSONDecodeError:
        pass

    return None, None, None

In [ ]:
# The model was fine-tuned on Multilingual-Thinking: it reasons in a target
# non-English language (set via the system prompt) and answers in English.
system_prompt = """
You are an AI assistant that thinks in {language} but responds in English.

IMPORTANT: Follow this exact format for every response:
1. First, write your reasoning and thoughts inside <think>...</think> tags
2. Then, provide your final answer in English

Always think through the problem in {language}, then translate your conclusion to English for the final response.
"""

prompt = "What is the capital of France, and why did it become the capital?"

In [ ]:
system = system_prompt.format(language="Italian")

request_body = {
    "messages": [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ],
    "max_tokens": 4096,
    "temperature": 0.3,
    "top_p": 0.9,
    "stream": True,
}

response = sagemaker_client.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    InferenceComponentName=ic_name,
    Body=json.dumps(request_body),
    ContentType="application/json",
)

reasoning_text = ""
generated_text = ""
tool_calls_data = []
in_reasoning = False

for line in LineIterator(response["Body"]):
    if not line:
        continue
    reasoning, content, tool_calls = parse_streaming_response(line.decode("utf-8"))
    if reasoning:
        if not in_reasoning:
            print("--- reasoning ---")
            in_reasoning = True
        reasoning_text += reasoning
        print(reasoning, end="", flush=True)
    if content:
        if in_reasoning:
            print("\n--- answer ---")
            in_reasoning = False
        generated_text += content
        print(content, end="", flush=True)
    if tool_calls:
        if in_reasoning:
            print("\n--- answer ---")
            in_reasoning = False
        tool_calls_data.extend(tool_calls)
        print(f"\n[tool_calls] {tool_calls}", flush=True)

***

### Step 8 – Clean up resources

> ⚠️ **Important:** SageMaker real-time endpoints accrue costs as long as they're running, even when idle. Run all cells below to delete every resource created in this lab.

Delete in this order to avoid dependency conflicts:

1. **Inference Component** — releases the GPU allocation and stops the vLLM server
2. **Model** — removes the model artifact reference (does not delete S3 data)
3. **Endpoint** — terminates the underlying compute instance (**this stops billing**)
4. **Endpoint Configuration** — removes the infrastructure blueprint

In [ ]:
from sagemaker.core.resources import InferenceComponent

# Delete inference component
InferenceComponent.get(inference_component_name=ic_name).delete()

In [ ]:
from sagemaker.core.resources import Model

# Delete model
Model.get(model_name=model_name).delete()

In [ ]:
from sagemaker.core.resources import Endpoint

# Delete endpoint (optional - if you want to remove the endpoint too)
Endpoint.get(endpoint_name=endpoint_name).delete()

In [ ]:
from sagemaker.core.resources import EndpointConfig

# Delete endpoint config (optional)
EndpointConfig.get(endpoint_config_name=endpoint_config_name).delete()